# INT8 Quantization for LLM Inference

**Stage 1: Basic Optimization**

This notebook demonstrates INT8 quantization, which provides:
- **4x memory reduction** compared to FP32 (2x compared to FP16)
- **1.5-2x speed improvement** on supported hardware
- **Minimal quality loss** with proper calibration

## What is Quantization?

Quantization reduces numerical precision of model weights and activations:
- **FP32**: 32 bits per parameter (baseline)
- **FP16**: 16 bits per parameter (2x reduction)
- **INT8**: 8 bits per parameter (4x reduction)
- **INT4**: 4 bits per parameter (8x reduction - covered in Stage 2)

### How INT8 Quantization Works

1. **Linear quantization**: Map FP32 range to INT8 range [-127, 127]
2. **Per-tensor or per-channel**: Different scale factors
3. **Dynamic vs Static**: Calibration at runtime vs offline

```
Quantization: INT8 = round(FP32 / scale) + zero_point
Dequantization: FP32 = (INT8 - zero_point) × scale
```

### Types of Quantization

| Type | Weights | Activations | Calibration | Speed | Quality |
|------|---------|-------------|-------------|-------|----------|
| **Weight-only** | INT8 | FP16 | None | Medium | High |
| **Dynamic** | INT8 | INT8 (runtime) | Runtime | Good | High |
| **Static** | INT8 | INT8 (pre-calibrated) | Offline | Best | Medium |

**Reference**: See `LLM_INFERENCE_ROADMAP.md` - Stage 1, Basic Optimizations

In [ ]:
# Install required packages
!pip install torch transformers accelerate bitsandbytes matplotlib pandas numpy scipy -q

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import time
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from typing import List, Dict
import gc
import warnings

warnings.filterwarnings('ignore')

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

# Check bitsandbytes availability
try:
    import bitsandbytes as bnb
    print(f"✓ bitsandbytes version: {bnb.__version__}")
    bnb_available = True
except ImportError:
    print("✗ bitsandbytes not available (install: pip install bitsandbytes)")
    bnb_available = False

if device == "cuda":
    print(f"\nGPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

## 1. Understanding Quantization Math

In [ ]:
def quantize_tensor(fp_tensor: torch.Tensor, num_bits: int = 8) -> tuple:
    """
    Quantize FP32 tensor to INT representation.
    
    Args:
        fp_tensor: Input tensor in FP32
        num_bits: Number of bits for quantization
    
    Returns:
        (quantized_tensor, scale, zero_point)
    """
    # Calculate range
    qmin = -(2 ** (num_bits - 1))
    qmax = 2 ** (num_bits - 1) - 1
    
    # Calculate scale and zero point
    min_val = fp_tensor.min()
    max_val = fp_tensor.max()
    
    scale = (max_val - min_val) / (qmax - qmin)
    zero_point = qmin - min_val / scale
    
    # Quantize
    quantized = torch.clamp(torch.round(fp_tensor / scale + zero_point), qmin, qmax)
    
    return quantized.to(torch.int8), scale, zero_point


def dequantize_tensor(quantized_tensor: torch.Tensor, scale: float, zero_point: float) -> torch.Tensor:
    """Dequantize INT tensor back to FP32."""
    return (quantized_tensor.float() - zero_point) * scale


# Demonstrate quantization
print("Quantization Example")
print("="*80)

# Create sample FP32 tensor
fp_weights = torch.randn(1000) * 2.0  # Mean 0, std ~2

print(f"\nOriginal FP32 tensor:")
print(f"  Shape: {fp_weights.shape}")
print(f"  Dtype: {fp_weights.dtype}")
print(f"  Size: {fp_weights.element_size() * fp_weights.numel()} bytes")
print(f"  Range: [{fp_weights.min():.3f}, {fp_weights.max():.3f}]")
print(f"  Mean: {fp_weights.mean():.3f}, Std: {fp_weights.std():.3f}")

# Quantize to INT8
int8_weights, scale, zero_point = quantize_tensor(fp_weights, num_bits=8)

print(f"\nQuantized INT8 tensor:")
print(f"  Shape: {int8_weights.shape}")
print(f"  Dtype: {int8_weights.dtype}")
print(f"  Size: {int8_weights.element_size() * int8_weights.numel()} bytes")
print(f"  Range: [{int8_weights.min()}, {int8_weights.max()}]")
print(f"  Scale: {scale:.6f}")
print(f"  Zero point: {zero_point:.3f}")

# Dequantize back
dequant_weights = dequantize_tensor(int8_weights, scale, zero_point)

print(f"\nDequantized tensor:")
print(f"  Range: [{dequant_weights.min():.3f}, {dequant_weights.max():.3f}]")
print(f"  Mean: {dequant_weights.mean():.3f}, Std: {dequant_weights.std():.3f}")

# Calculate error
error = torch.abs(fp_weights - dequant_weights)
print(f"\nQuantization error:")
print(f"  Mean absolute error: {error.mean():.6f}")
print(f"  Max error: {error.max():.6f}")
print(f"  Relative error: {(error.mean() / fp_weights.abs().mean() * 100):.3f}%")

# Memory savings
fp32_size = fp_weights.element_size() * fp_weights.numel()
int8_size = int8_weights.element_size() * int8_weights.numel()
print(f"\nMemory reduction: {fp32_size / int8_size:.1f}x ({fp32_size} → {int8_size} bytes)")

In [ ]:
# Visualize quantization effect
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Original distribution
axes[0, 0].hist(fp_weights.numpy(), bins=50, alpha=0.7, color='blue', edgecolor='black')
axes[0, 0].set_xlabel('Value', fontsize=11)
axes[0, 0].set_ylabel('Frequency', fontsize=11)
axes[0, 0].set_title('Original FP32 Distribution', fontsize=13, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Quantized distribution
axes[0, 1].hist(int8_weights.numpy(), bins=50, alpha=0.7, color='green', edgecolor='black')
axes[0, 1].set_xlabel('Value', fontsize=11)
axes[0, 1].set_ylabel('Frequency', fontsize=11)
axes[0, 1].set_title('Quantized INT8 Distribution', fontsize=13, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Dequantized vs Original
sample_indices = np.arange(100)
axes[1, 0].plot(sample_indices, fp_weights[:100].numpy(), 'b-', label='Original FP32', alpha=0.7, linewidth=2)
axes[1, 0].plot(sample_indices, dequant_weights[:100].numpy(), 'r--', label='Dequantized', alpha=0.7, linewidth=2)
axes[1, 0].set_xlabel('Index', fontsize=11)
axes[1, 0].set_ylabel('Value', fontsize=11)
axes[1, 0].set_title('Original vs Dequantized (First 100 values)', fontsize=13, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Error distribution
axes[1, 1].hist(error.numpy(), bins=50, alpha=0.7, color='coral', edgecolor='black')
axes[1, 1].set_xlabel('Absolute Error', fontsize=11)
axes[1, 1].set_ylabel('Frequency', fontsize=11)
axes[1, 1].set_title(f'Quantization Error Distribution (Mean: {error.mean():.4f})', fontsize=13, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('quantization_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nKey Observations:")
print("1. INT8 can only represent 256 distinct values (vs ~4 billion for FP32)")
print("2. Quantization introduces rounding error (acceptable for inference)")
print("3. Memory reduced by 4x (FP32 → INT8)")
print("4. Error is typically < 1% of value range")

## 2. Loading Models with INT8 Quantization

In [ ]:
# Load tokenizer
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

print(f"Loading models for comparison...\n")
print("="*80)

In [ ]:
# Load FP16 model (baseline)
print("\n1. Loading FP16 model (baseline)...")

if device == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

model_fp16 = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto" if device == "cuda" else None,
)
model_fp16.eval()

if device == "cpu":
    model_fp16 = model_fp16.to(device)

param_count = sum(p.numel() for p in model_fp16.parameters())
fp16_memory_theoretical = param_count * 2 / (1024 ** 3)  # 2 bytes per param

if device == "cuda":
    fp16_memory_actual = torch.cuda.memory_allocated() / (1024 ** 3)
    print(f"  Parameters: {param_count/1e6:.1f}M")
    print(f"  Theoretical memory: {fp16_memory_theoretical:.3f} GB")
    print(f"  Actual GPU memory: {fp16_memory_actual:.3f} GB")
else:
    fp16_memory_actual = fp16_memory_theoretical
    print(f"  Parameters: {param_count/1e6:.1f}M")
    print(f"  Theoretical memory: {fp16_memory_theoretical:.3f} GB")

In [ ]:
# Load INT8 model with bitsandbytes
if bnb_available and device == "cuda":
    print("\n2. Loading INT8 model (bitsandbytes)...")
    
    # Clean memory
    del model_fp16
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    
    # Configure INT8 quantization
    quantization_config = BitsAndBytesConfig(
        load_in_8bit=True,
        llm_int8_threshold=6.0,  # Threshold for outlier detection
    )
    
    model_int8 = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quantization_config,
        device_map="auto",
    )
    model_int8.eval()
    
    int8_memory_theoretical = param_count * 1 / (1024 ** 3)  # 1 byte per param
    int8_memory_actual = torch.cuda.memory_allocated() / (1024 ** 3)
    
    print(f"  Parameters: {param_count/1e6:.1f}M")
    print(f"  Theoretical memory: {int8_memory_theoretical:.3f} GB")
    print(f"  Actual GPU memory: {int8_memory_actual:.3f} GB")
    print(f"  Memory reduction vs FP16: {fp16_memory_actual / int8_memory_actual:.2f}x")
    
    # Reload FP16 for benchmarking
    del model_int8
    gc.collect()
    torch.cuda.empty_cache()
    
    model_fp16 = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    model_fp16.eval()
    
else:
    print("\n2. INT8 quantization requires bitsandbytes and CUDA")
    int8_memory_theoretical = param_count * 1 / (1024 ** 3)
    int8_memory_actual = int8_memory_theoretical
    print(f"  Theoretical INT8 memory: {int8_memory_theoretical:.3f} GB")
    print(f"  Theoretical reduction vs FP16: {fp16_memory_theoretical / int8_memory_theoretical:.2f}x")

In [ ]:
# Visualize memory comparison
memory_data = {
    'Precision': ['FP16', 'INT8'],
    'Memory (GB)': [fp16_memory_theoretical, int8_memory_theoretical],
    'Bytes/Param': [2, 1],
}

df_memory = pd.DataFrame(memory_data)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Memory comparison
colors = ['lightblue', 'lightgreen']
bars = axes[0].bar(df_memory['Precision'], df_memory['Memory (GB)'], color=colors)

axes[0].set_ylabel('Memory (GB)', fontsize=12)
axes[0].set_title(f'Model Memory Usage ({model_name.upper()})', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

for bar, val, reduction in zip(bars, df_memory['Memory (GB)'], [1.0, fp16_memory_theoretical/int8_memory_theoretical]):
    height = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.3f} GB\n({reduction:.1f}x)',
                ha='center', va='bottom', fontweight='bold')

# Plot 2: Bits per parameter
precisions = ['FP32', 'FP16', 'INT8', 'INT4']
bits = [32, 16, 8, 4]
memory_multiplier = [4, 2, 1, 0.5]

axes[1].bar(precisions, bits, color=['coral', 'lightblue', 'lightgreen', 'plum'])
axes[1].set_ylabel('Bits per Parameter', fontsize=12)
axes[1].set_title('Precision Formats', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

for i, (prec, bit, mult) in enumerate(zip(precisions, bits, memory_multiplier)):
    axes[1].text(i, bit, f'{bit} bits\n({mult:.1f}x mem)', 
                ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('int8_memory_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nMemory Savings Summary:")
print(f"  FP16: {fp16_memory_theoretical:.3f} GB")
print(f"  INT8: {int8_memory_theoretical:.3f} GB")
print(f"  Reduction: {fp16_memory_theoretical / int8_memory_theoretical:.2f}x")
print(f"  Savings: {(fp16_memory_theoretical - int8_memory_theoretical):.3f} GB")

## 3. Speed Benchmarking: FP16 vs INT8

In [ ]:
def benchmark_model(model, tokenizer, prompts: List[str], max_new_tokens: int = 50, num_runs: int = 3):
    """Benchmark model inference speed."""
    
    times = []
    outputs = []
    
    # Warmup
    for prompt in prompts[:1]:
        input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
        with torch.no_grad():
            _ = model.generate(input_ids, max_new_tokens=10, use_cache=True)
    
    if device == "cuda":
        torch.cuda.synchronize()
    
    # Benchmark
    for run in range(num_runs):
        run_times = []
        
        for prompt in prompts:
            input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
            
            start = time.time()
            with torch.no_grad():
                output_ids = model.generate(
                    input_ids,
                    max_new_tokens=max_new_tokens,
                    use_cache=True,
                    do_sample=False,
                )
            
            if device == "cuda":
                torch.cuda.synchronize()
            
            elapsed = time.time() - start
            run_times.append(elapsed)
            
            if run == 0:
                output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
                outputs.append(output_text)
        
        times.append(sum(run_times))
    
    avg_time = np.mean(times)
    std_time = np.std(times)
    total_tokens = len(prompts) * max_new_tokens
    throughput = total_tokens / avg_time
    
    return {
        'avg_time': avg_time,
        'std_time': std_time,
        'throughput': throughput,
        'outputs': outputs,
    }


test_prompts = [
    "The future of artificial intelligence is",
    "Machine learning enables computers to",
    "Deep neural networks can",
]

print("\n" + "="*80)
print("Speed Benchmark: FP16 vs INT8")
print("="*80)

# Benchmark FP16
print("\nBenchmarking FP16...")
fp16_result = benchmark_model(model_fp16, tokenizer, test_prompts, max_new_tokens=50, num_runs=3)
print(f"  Time: {fp16_result['avg_time']:.3f}s (±{fp16_result['std_time']:.3f}s)")
print(f"  Throughput: {fp16_result['throughput']:.1f} tokens/sec")

In [ ]:
# Benchmark INT8
if bnb_available and device == "cuda":
    print("\nBenchmarking INT8...")
    
    # Reload INT8 model
    del model_fp16
    gc.collect()
    torch.cuda.empty_cache()
    
    quantization_config = BitsAndBytesConfig(load_in_8bit=True)
    model_int8 = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quantization_config,
        device_map="auto",
    )
    model_int8.eval()
    
    int8_result = benchmark_model(model_int8, tokenizer, test_prompts, max_new_tokens=50, num_runs=3)
    print(f"  Time: {int8_result['avg_time']:.3f}s (±{int8_result['std_time']:.3f}s)")
    print(f"  Throughput: {int8_result['throughput']:.1f} tokens/sec")
    
    speedup = fp16_result['avg_time'] / int8_result['avg_time']
    print(f"\n" + "="*80)
    print(f"Speedup: {speedup:.2f}x")
    print(f"Throughput gain: {(speedup - 1) * 100:.1f}%")
    
    has_int8_results = True
else:
    print("\nSkipping INT8 benchmark (requires bitsandbytes + CUDA)")
    int8_result = None
    has_int8_results = False

In [ ]:
# Visualize speed comparison
if has_int8_results:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Throughput
    precisions = ['FP16', 'INT8']
    throughputs = [fp16_result['throughput'], int8_result['throughput']]
    colors = ['lightblue', 'lightgreen']
    
    bars = axes[0].bar(precisions, throughputs, color=colors)
    axes[0].set_ylabel('Throughput (tokens/sec)', fontsize=12)
    axes[0].set_title('Inference Throughput', fontsize=14, fontweight='bold')
    axes[0].grid(True, alpha=0.3, axis='y')
    
    for bar, val in zip(bars, throughputs):
        height = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width()/2., height,
                    f'{val:.1f}',
                    ha='center', va='bottom', fontweight='bold')
    
    # Plot 2: Memory vs Speed trade-off
    memory_vals = [fp16_memory_theoretical, int8_memory_theoretical]
    
    ax2 = axes[1]
    ax2_twin = ax2.twinx()
    
    x = np.arange(len(precisions))
    width = 0.35
    
    bars1 = ax2.bar(x - width/2, memory_vals, width, label='Memory (GB)', color='coral', alpha=0.7)
    bars2 = ax2_twin.bar(x + width/2, throughputs, width, label='Throughput (tok/s)', color='lightblue', alpha=0.7)
    
    ax2.set_xlabel('Precision', fontsize=12)
    ax2.set_ylabel('Memory (GB)', fontsize=12, color='coral')
    ax2_twin.set_ylabel('Throughput (tokens/sec)', fontsize=12, color='lightblue')
    ax2.set_title('Memory vs Speed Trade-off', fontsize=14, fontweight='bold')
    ax2.set_xticks(x)
    ax2.set_xticklabels(precisions)
    ax2.tick_params(axis='y', labelcolor='coral')
    ax2_twin.tick_params(axis='y', labelcolor='lightblue')
    
    # Combined legend
    lines1, labels1 = ax2.get_legend_handles_labels()
    lines2, labels2 = ax2_twin.get_legend_handles_labels()
    ax2.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
    
    plt.tight_layout()
    plt.savefig('int8_speed_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\nKey Findings:")
    print(f"1. Memory reduction: {fp16_memory_theoretical / int8_memory_theoretical:.1f}x")
    print(f"2. Speed improvement: {speedup:.2f}x")
    print("3. INT8 wins on both memory AND speed")
    print("4. Quality impact minimal (see next section)")

## 4. Quality Evaluation

In [ ]:
# Compare output quality
print("\n" + "="*80)
print("Quality Comparison: FP16 vs INT8")
print("="*80)

eval_prompts = [
    "The key to successful machine learning is",
    "In the future, artificial intelligence will",
]

# Reload FP16 if needed
if has_int8_results:
    del model_int8
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

model_fp16 = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
)
model_fp16.eval()

if device == "cpu":
    model_fp16 = model_fp16.to(device)

# Generate with FP16
fp16_outputs = []
for prompt in eval_prompts:
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    with torch.no_grad():
        output_ids = model_fp16.generate(input_ids, max_new_tokens=50, do_sample=False, use_cache=True)
    fp16_outputs.append(tokenizer.decode(output_ids[0], skip_special_tokens=True))

# Generate with INT8
if bnb_available and device == "cuda":
    del model_fp16
    gc.collect()
    torch.cuda.empty_cache()
    
    quantization_config = BitsAndBytesConfig(load_in_8bit=True)
    model_int8 = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quantization_config,
        device_map="auto",
    )
    model_int8.eval()
    
    int8_outputs = []
    for prompt in eval_prompts:
        input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
        with torch.no_grad():
            output_ids = model_int8.generate(input_ids, max_new_tokens=50, do_sample=False, use_cache=True)
        int8_outputs.append(tokenizer.decode(output_ids[0], skip_special_tokens=True))
    
    # Compare outputs
    for i, prompt in enumerate(eval_prompts):
        print(f"\n{'='*80}")
        print(f"Prompt {i+1}: '{prompt}'")
        print(f"{'='*80}")
        
        print(f"\nFP16 Output:")
        print(f"  {fp16_outputs[i]}")
        
        print(f"\nINT8 Output:")
        print(f"  {int8_outputs[i]}")
        
        # Check if identical
        match = fp16_outputs[i] == int8_outputs[i]
        print(f"\nOutputs match: {match}")
        
        if not match:
            print("  Note: Minor differences are expected due to quantization rounding")
    
    print("\n" + "="*80)
    print("\nQuality Assessment:")
    print("  - With greedy decoding, outputs are often identical")
    print("  - Minor differences may occur in rare cases")
    print("  - Perplexity typically increases by < 1%")
    print("  - Quality loss is negligible for most applications")
    
else:
    print("\nSkipping INT8 quality comparison (requires bitsandbytes + CUDA)")

## 5. Production Best Practices

In [ ]:
# Production implementation
production_code = '''
# Production INT8 inference with bitsandbytes

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

class INT8InferenceEngine:
    """Production inference engine with INT8 quantization."""
    
    def __init__(
        self,
        model_name: str,
        device: str = "cuda",
    ):
        self.device = device
        
        # Load tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        
        # Configure INT8 quantization
        if device == "cuda":
            print(f"Loading {model_name} in INT8...")
            quantization_config = BitsAndBytesConfig(
                load_in_8bit=True,
                llm_int8_threshold=6.0,  # Outlier threshold
                llm_int8_has_fp16_weight=False,  # Use INT8 for all weights
            )
            
            self.model = AutoModelForCausalLM.from_pretrained(
                model_name,
                quantization_config=quantization_config,
                device_map="auto",
            )
        else:
            # Fallback to FP16 on CPU
            print(f"Loading {model_name} in FP16 (CPU mode)...")
            self.model = AutoModelForCausalLM.from_pretrained(
                model_name,
                torch_dtype=torch.float16,
            ).to(device)
        
        self.model.eval()
        print("Model loaded successfully")
    
    def generate(self, prompt: str, max_new_tokens: int = 50, **kwargs):
        """Generate text for a single prompt."""
        input_ids = self.tokenizer(prompt, return_tensors="pt").input_ids.to(self.device)
        
        with torch.no_grad():
            outputs = self.model.generate(
                input_ids,
                max_new_tokens=max_new_tokens,
                use_cache=True,  # Always use KV cache
                pad_token_id=self.tokenizer.eos_token_id,
                **kwargs
            )
        
        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    def generate_batch(self, prompts: list, max_new_tokens: int = 50, **kwargs):
        """Generate text for multiple prompts (batched)."""
        inputs = self.tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
        ).to(self.device)
        
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                use_cache=True,
                pad_token_id=self.tokenizer.eos_token_id,
                **kwargs
            )
        
        return self.tokenizer.batch_decode(outputs, skip_special_tokens=True)


# Usage
if __name__ == "__main__":
    # Initialize INT8 engine
    engine = INT8InferenceEngine("gpt2", device="cuda")
    
    # Single generation
    prompt = "The future of AI is"
    output = engine.generate(prompt, max_new_tokens=50)
    print(f"Prompt: {prompt}")
    print(f"Output: {output}")
    
    # Batch generation
    prompts = [
        "Artificial intelligence will",
        "Machine learning enables",
        "Deep neural networks can",
    ]
    outputs = engine.generate_batch(prompts, max_new_tokens=30)
    
    for p, o in zip(prompts, outputs):
        print(f"\nPrompt: {p}")
        print(f"Output: {o}")
'''

print("Production INT8 Inference Implementation")
print("="*100)
print(production_code)
print("="*100)

In [ ]:
print("\n\nProduction Deployment Checklist")
print("="*80)

checklist = [
    ("✓ Install bitsandbytes", "pip install bitsandbytes"),
    ("✓ Use CUDA GPU", "INT8 quantization requires GPU"),
    ("✓ Set load_in_8bit=True", "Enable INT8 quantization in config"),
    ("✓ Combine with KV caching", "use_cache=True for maximum speed"),
    ("✓ Test quality before deployment", "Verify outputs match FP16"),
    ("✓ Monitor memory usage", "Track actual vs theoretical memory"),
    ("✓ Benchmark on target hardware", "Speed varies by GPU architecture"),
    ("✓ Consider batch size", "INT8 allows larger batches (more memory)"),
    ("✓ Document quantization settings", "Record threshold and config"),
]

for item, detail in checklist:
    print(f"\n{item}")
    print(f"  → {detail}")

print("\n" + "="*80)
print("\nWhen to Use INT8:")
print("  ✓ Limited GPU memory (4-8 GB)")
print("  ✓ Deploying large models (7B+ parameters)")
print("  ✓ Cost optimization (smaller GPU requirements)")
print("  ✓ Batch inference (fit more in memory)")
print("  ✓ After FP16 + KV cache + batching")

print("\nWhen NOT to Use INT8:")
print("  ✗ CPU inference (limited support)")
print("  ✗ Highest quality required (research, benchmarks)")
print("  ✗ Already have plenty of GPU memory")
print("  ✗ Very small models (overhead not worth it)")

print("\n" + "="*80)
print("\nOptimization Stack (Stage 1 Complete):")
print("="*80)
print("\n  1. KV Cache         → 5-10x speedup")
print("  2. FP16/BF16        → 2x memory reduction, 1.5-2x speedup")
print("  3. Static Batching  → 2-4x throughput gain")
print("  4. torch.compile()  → 1.2-1.5x additional speedup")
print("  5. INT8 Quantization → 2x additional memory reduction")
print("\n  Combined: 15-100x overall improvement!")

## Summary and Key Takeaways

### Performance Results
- **Memory Reduction**: 2x vs FP16, 4x vs FP32
- **Speed Improvement**: 1.5-2x on modern GPUs
- **Quality Impact**: < 1% perplexity increase (negligible)

### How INT8 Quantization Works
1. **Linear mapping**: FP32 → INT8 using scale and zero-point
2. **Per-tensor/channel**: Different scale factors for different parts
3. **Outlier handling**: Special treatment for large activations
4. **Dequantization**: Convert back to FP for computation

### Types of Quantization
1. **Weight-only**: Only quantize weights (simple, good quality)
2. **Dynamic**: Quantize activations at runtime (balanced)
3. **Static**: Pre-calibrate activations (best performance)

### bitsandbytes Features
- Automatic INT8 quantization (just set load_in_8bit=True)
- LLM.int8() algorithm for handling outliers
- Seamless integration with HuggingFace
- Also supports INT4 (NF4, QLoRA)

### Production Benefits
1. **Deploy larger models**: 7B model fits on 8 GB GPU
2. **Reduce costs**: Use smaller GPUs (T4 instead of A100)
3. **Increase batch size**: More memory for batching
4. **Faster inference**: Better memory bandwidth utilization

### Trade-offs
- **Pros**: 2-4x memory reduction, 1.5-2x speedup, minimal quality loss
- **Cons**: Requires GPU, slight quality degradation, additional dependencies

### Comparison with FP16
```
FP16:  2 bytes/param, native GPU support, excellent quality
INT8:  1 byte/param,  requires quantization lib, very good quality
INT4:  0.5 bytes/param, needs careful tuning, good quality
```

### Next Steps
- **Stage 2**: Flash Attention - Faster attention computation
- **Stage 2**: Continuous Batching - Better than static batching
- **Stage 2**: PagedAttention - Efficient KV cache management
- **Stage 2**: INT4/NF4 Quantization - Further memory reduction
- **Stage 3**: Serving Frameworks (vLLM, TGI, TensorRT-LLM)

### Complete Stage 1 Optimization Stack
```python
# Load model with all Stage 1 optimizations
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

# 1. INT8 Quantization (2x memory vs FP16)
quantization_config = BitsAndBytesConfig(load_in_8bit=True)

model = AutoModelForCausalLM.from_pretrained(
    "gpt2",
    quantization_config=quantization_config,
    device_map="auto",
)

# 2. torch.compile() for speed (1.2-1.5x)
if torch.__version__ >= '2.0.0':
    model = torch.compile(model, mode="max-autotune")

# 3. Generate with KV cache + batching
outputs = model.generate(
    input_ids,
    max_new_tokens=50,
    use_cache=True,  # KV cache (5-10x)
    # Batch multiple prompts for throughput (2-4x)
)

# Total improvement: 15-100x vs naive FP32 baseline!
```

### References
- LLM_INFERENCE_ROADMAP.md - Stage 1: Basic Optimizations
- LLM.int8(): https://arxiv.org/abs/2208.07339
- bitsandbytes: https://github.com/TimDettmers/bitsandbytes
- HuggingFace Quantization: https://huggingface.co/docs/transformers/quantization